**Status: live in-race podium and winner chances (replaces `live_model` from `phase_3_ranking.ipynb`).**

The old lap-level ranker was no better than the running order (top-3 0.791 vs 0.799 on 2022-23). This notebook has two models, each judged against that baseline:

- **Podium (early-race hybrid):** up to 50% race distance, a shallow XGBoost on the pre-race podium probability (`f1_podium_prior_oof`) plus lap state (position, places lost or gained since the grid, gaps to P3/P4, pit stops still owed, recent pace, tyre age, safety car). After 50% it uses the running order, which every model lost to late in the race.
- **Winner:** a logistic model over the top 4 cars (pre-race probability, gaps, closing rate, pace and tyre-age difference to the car ahead, pit stops owed, laps left, safety car), scored against "the current leader wins".

Rules fixed in advance (design chosen on 2022-23, tested once on 2024-26): podium ships if the race-level 95% CI of the 0-50% top-3 gain over the running order is above zero (passed: +2.5 pts [+0.8, +4.3]); winner ships if the 70-100% gain CI is above zero (failed: +2.8 [-0.5, +6.8]; mid-race +4 to +5 pts was significant). The walk-forward backtest below re-measures both on every season from 2022 and stores the flags; the App shows the winner chance as experimental whatever the flag says.

Predictions are walk-forward by season (trained only on earlier seasons), from 2021. Writes `f1_live_predictions` and `f1_live_backtest`.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier

FIRST_TRAIN_YEAR = 2020
EARLY_CUT = 0.5
EARLY_TRAIN_MAX = 0.7
K = ["Year", "Race", "LapNumber"]
D = ["Year", "Race", "Driver"]

In [ ]:
cols = ["Year", "Race", "Driver", "LapNumber", "Position", "LapTime", "Time", "Stint", "TyreLife", "TrackStatus", "PitInTime", "PitOutTime"]
laps = spark.table("workspace.default.f1_master_lap_dataset").select(*cols).toPandas()
prior = spark.table("workspace.default.f1_podium_prior_oof").toPandas()
d = laps[laps["Year"] >= FIRST_TRAIN_YEAR].dropna(subset=["Position", "Time"]).drop_duplicates(D + ["LapNumber"])
d = d.merge(prior[["Year", "Race", "Driver", "Prior", "GridPosition", "FinalPosition"]], on=D, how="inner")
d["Podium"] = (d["FinalPosition"] <= 3).astype(int)
d["Win"] = (d["FinalPosition"] == 1).astype(int)

d = d.sort_values(K + ["Position"]).reset_index(drop=True)
d["Rank"] = d.groupby(K)["Position"].rank(method="first")
d["DeltaToLeader"] = d["Time"] - d.groupby(K)["Time"].transform("min")
d = d.sort_values(K + ["Rank"]).reset_index(drop=True)
d["GapToAhead"] = d.groupby(K)["Time"].diff()
d["GapBehind"] = d.groupby(K)["GapToAhead"].shift(-1)
d["RaceLaps"] = d.groupby(["Year", "Race"])["LapNumber"].transform("max")
d["LapFrac"], d["LapsLeft"] = d["LapNumber"] / d["RaceLaps"], d["RaceLaps"] - d["LapNumber"]
d["Stops"] = d["Stint"].fillna(1) - 1
d["SC"] = d["TrackStatus"].astype(str).str.contains("[45]").astype(int)
d["TyreLife"] = d["TyreLife"].fillna(0)
d["PriorLogit"] = np.log(d["Prior"].clip(1e-3, 1 - 1e-3) / (1 - d["Prior"].clip(1e-3, 1 - 1e-3)))
d["GridDelta"] = (d["Rank"] - d["GridPosition"].where(d["GridPosition"] > 0, 20).fillna(20)).clip(-19, 19)

# recent pace: median of the last 5 clean laps (no pit in/out, no SC), relative to the leader
d = d.sort_values(D + ["LapNumber"]).reset_index(drop=True)
pit = d["PitInTime"].notna() | d["PitOutTime"].notna()
clean = d["LapTime"].where(~pit & (d["SC"] == 0) & (d["LapNumber"] > 1))
d["Pace"] = clean.groupby([d["Year"], d["Race"], d["Driver"]]).transform(lambda s: s.rolling(5, min_periods=2).median())
d["Pace"] = d.groupby(D)["Pace"].ffill()
d["Closing"] = (d.groupby(D)["GapToAhead"].shift(3) - d["GapToAhead"]) / 3
d = d.join(d[d["Rank"] == 1].set_index(K)["Pace"].rename("LeadPace"), on=K)
d["RelPace"] = (d["Pace"] - d["LeadPace"]).clip(-2, 2).fillna(0)
d = d.join(d[d["Rank"] <= 6].groupby(K)["Stops"].median().rename("TopStops"), on=K)
d["PitDebt"] = (d["TopStops"] - d["Stops"]).clip(-1, 2)
for r in (3, 4):
    d = d.join(d[d["Rank"] == r].set_index(K)["DeltaToLeader"].rename(f"D{r}"), on=K)
d["GapToP3"], d["GapToP4"] = d["DeltaToLeader"] - d["D3"], d["D4"] - d["DeltaToLeader"]
d = d.sort_values(K + ["Rank"]).reset_index(drop=True)
d["TyreDeltaAhead"] = d["TyreLife"] - d.groupby(K)["TyreLife"].shift(1)
d["PaceDeltaAhead"] = d["RelPace"] - d.groupby(K)["RelPace"].shift(1)
print("Races per season:", d.groupby("Year")["Race"].nunique().to_dict())

In [ ]:
EARLY = ["PriorLogit", "Rank", "GridDelta", "LapFrac", "GapToP3", "GapToP4", "GapBehind", "PitDebt", "Stops", "TyreLife", "SC", "RelPace"]

def fit_podium(tr):
    tr = tr[tr["LapFrac"] <= EARLY_TRAIN_MAX]
    return XGBClassifier(n_estimators=300, max_depth=2, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8,
                         min_child_weight=50, eval_metric="logloss", random_state=42).fit(tr[EARLY], tr["Podium"])

def win_x(x):
    return np.nan_to_num(np.c_[x["PriorLogit"], np.log(x["Rank"]), np.log(x["Rank"]) * x["LapFrac"], x["GapToAhead"].clip(0, 30),
                               x["GapToAhead"].clip(0, 30) / (x["LapsLeft"] + 1), x["Closing"].clip(-3, 3) * x["LapsLeft"] / 10,
                               x["PaceDeltaAhead"] * x["LapsLeft"] / 10, x["TyreDeltaAhead"] / 10, x["PitDebt"], x["SC"]])

def fit_win(tr):
    tr = tr[tr["Rank"] <= 4]
    return LogisticRegression(C=1.0, max_iter=5000).fit(win_x(tr), tr["Win"])

def running_lookup(tr):
    # historical podium rate by running position and race-distance decile: the probability behind "trust the running order"
    b = lambda x: np.minimum((x["LapFrac"] * 10).astype(int), 9)
    return tr.groupby([tr["Rank"].clip(upper=20), b(tr)])["Podium"].mean(), b

def norm_to(p, total):
    # shift logits by one constant so a lap's probabilities sum to `total` (3 for podium, 1 for winner)
    z = np.log(np.clip(p, 1e-4, 1 - 1e-4) / (1 - np.clip(p, 1e-4, 1 - 1e-4)))
    lo, hi = -15.0, 15.0
    for _ in range(50):
        mid = (lo + hi) / 2
        lo, hi = (lo, mid) if (1 / (1 + np.exp(-(z + mid)))).sum() > total else (mid, hi)
    return 1 / (1 + np.exp(-(z + (lo + hi) / 2)))

In [ ]:
out = []
for year in range(FIRST_TRAIN_YEAR + 1, int(d["Year"].max()) + 1):
    tr, te = d[d["Year"] < year], d[d["Year"] == year].copy()
    pod, win = fit_podium(tr), fit_win(tr)
    tab, bucket = running_lookup(tr)
    te["PodiumModel"] = pod.predict_proba(te[EARLY])[:, 1]
    te["PodiumRunning"] = [tab.get((min(r, 20), b), 0.0) for r, b in zip(te["Rank"], bucket(te))]
    te["WinRaw"] = np.where(te["Rank"] <= 4, win.predict_proba(win_x(te))[:, 1], 0.0)
    te["PodiumP"], te["WinP"] = np.nan, 0.0
    for _, idx in te.groupby(K).groups.items():
        g = te.loc[idx]
        early = g["LapFrac"].iloc[0] <= EARLY_CUT
        te.loc[idx, "PodiumP"] = norm_to(g["PodiumModel" if early else "PodiumRunning"].to_numpy(), min(3, len(g)))
        top = g["Rank"] <= 4
        te.loc[g.index[top], "WinP"] = norm_to(g.loc[top, "WinRaw"].to_numpy(), 1)
    te["PodiumSource"] = np.where(te["LapFrac"] <= EARLY_CUT, "model", "running order")
    out.append(te)
    print(year, "done:", te["Race"].nunique(), "races")
pred = pd.concat(out, ignore_index=True)

In [ ]:
def top_hits(df, score, label, k):
    s = df.assign(_s=np.asarray(score)).sort_values(K + ["_s"], ascending=[True, True, True, False])
    return (s.groupby(K).head(k).groupby(K)[label].sum() / k).groupby(level=["Year", "Race"]).mean()

def compare(df, score, label, k, lo, hi):
    w = df[(df["LapFrac"] > lo) & (df["LapFrac"] <= hi)]
    sc = np.asarray(score)[((df["LapFrac"] > lo) & (df["LapFrac"] <= hi)).to_numpy()]
    m, b = top_hits(w, sc, label, k), top_hits(w, -w["Rank"], label, k)
    g = (m - b).to_numpy()
    rng = np.random.default_rng(0)
    boot = [g[rng.integers(0, len(g), len(g))].mean() for _ in range(2000)]
    return dict(Model=m.mean(), RunningOrder=b.mean(), Gain=g.mean(), CI_Lo=np.percentile(boot, 2.5), CI_Hi=np.percentile(boot, 97.5), Races=len(g))

bt_src = pred[pred["Year"] >= 2022]
rows = []
for name, score, label, k in (("Podium", bt_src["PodiumP"], "Podium", 3), ("Winner", bt_src["WinP"] + (bt_src["Rank"] == 1) * 1e-6, "Win", 1)):
    for lo, hi in ((0, 0.5), (0.5, 1.0), (0.7, 1.0), (0, 1.0)):
        rows.append(dict(Target=name, Window=f"{int(lo * 100)}-{int(hi * 100)}%", **compare(bt_src, score, label, k, lo, hi)))
backtest = pd.DataFrame(rows)
SHIP_PODIUM = bool(backtest.query("Target == 'Podium' and Window == '0-50%'")["CI_Lo"].iloc[0] > 0)
SHIP_WINNER = bool(backtest.query("Target == 'Winner' and Window == '70-100%'")["CI_Lo"].iloc[0] > 0)
backtest["Ship"] = np.where(backtest["Target"] == "Podium", SHIP_PODIUM, SHIP_WINNER)
print(backtest.round(4).to_string(index=False))
print("Podium ship rule (0-50% CI above zero):", SHIP_PODIUM, "| Winner ship rule (70-100% CI above zero):", SHIP_WINNER)

In [ ]:
keep = ["Year", "Race", "LapNumber", "Driver", "Rank", "PodiumP", "WinP", "PodiumSource"]
spark.createDataFrame(pred[keep]).write.mode("overwrite").saveAsTable("workspace.default.f1_live_predictions")
spark.createDataFrame(backtest).write.mode("overwrite").saveAsTable("workspace.default.f1_live_backtest")
print("Wrote", len(pred), "rows; seasons", sorted(pred["Year"].unique()))